# Notebook 02 - Model Training

**Project:** Network Guardian IDS
**Phase:** 3 of 4
**Sprint Day:** 5 of 10

This notebook runs the three training pipelines. All logic lives in
`src/models.py`. This notebook only orchestrates and reports.

**Do not modify any cell in this notebook.** If a cell needs a change,
raise a decision file per `docs/decision/Decision Template.md`.

Cell ownership:

| Cell | Content | Owner |
|---|---|---|
| 2-4 | Shared setup | All |
| 5 | Random Forest training | Amber (Role 3) |
| 6 | Random Forest verification | Amber (Role 3) |
| 7 | XGBoost training | Kenneth (Role 4) |
| 8 | XGBoost verification | Kenneth (Role 4) |
| 9 | TabNet training | Dwayne (Role 5) |
| 10 | TabNet verification | Dwayne (Role 5) |
| 11 | Cross-model comparison | Kit (Role 1) |
| 12 | Summary | Kit (Role 1) |


In [2]:
import os
import sys
from pathlib import Path

_cwd = Path(os.getcwd()).resolve()
_repo_root = _cwd.parent if _cwd.name == "notebooks" else _cwd
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

print("Repo root on sys.path:", _repo_root)
print("src package present  :", (_repo_root / "src" / "__init__.py").is_file())


Repo root on sys.path: C:\Users\Rye\OneDrive\Desktop\network-guardian-ids
src package present  : True


In [4]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from src.models import (
    load_processed_arrays,
    encode_labels,
    train_rf,
    train_xgb,
    train_tabnet,
)
from src.utils import PATHS, RANDOM_STATE, set_seed

set_seed(RANDOM_STATE)

print("Repository root :", PATHS.repo_root)
print("Processed dir   :", PATHS.processed)
print("Models dir      :", PATHS.models)
print("Outputs dir     :", PATHS.outputs)
print("Seed            :", RANDOM_STATE)

X_train, y_train_str, X_test, y_test_str, class_order = load_processed_arrays()
y_train = encode_labels(y_train_str, class_order)
y_test  = encode_labels(y_test_str,  class_order)

print()
print("X_train    :", X_train.shape, X_train.dtype)
print("X_test     :", X_test.shape, X_test.dtype)
print("y_train    :", y_train.shape, y_train.dtype)
print("y_test     :", y_test.shape, y_test.dtype)
print("class_order:", class_order)


Repository root : C:\Users\Rye\OneDrive\Desktop\network-guardian-ids
Processed dir   : C:\Users\Rye\OneDrive\Desktop\network-guardian-ids\data\processed
Models dir      : C:\Users\Rye\OneDrive\Desktop\network-guardian-ids\models
Outputs dir     : C:\Users\Rye\OneDrive\Desktop\network-guardian-ids\outputs
Seed            : 42

X_train    : (109188, 43) float64
X_test     : (36396, 43) float64
y_train    : (109188,) int64
y_test     : (36396,) int64
class_order: ['normal', 'dos', 'probe', 'r2l', 'u2r']


In [3]:
assert class_order == ["normal", "dos", "probe", "r2l", "u2r"], (
    f"Class order mismatch: {class_order}"
)
print("Class order verified. Ready for training.")


Class order verified. Ready for training.


In [5]:
# Cell 5: Random Forest training (Amber, Role 3)
# Contract: docs/contracts/role-3-rf-architecture-spec.md
# Budget: N_TRIALS_RF = 20 trials, N_FOLDS_RF = 5 folds.
#
# This cell trains the RF baseline, refits once on the full cleaned pool,
# and persists both the model and its meta sidecar. The test partition is
# loaded only for a shape assertion and is then dropped from scope. No
# training function in this cell accepts the test partition.

from src.models import load_processed_arrays, encode_labels, train_rf

X_train, y_train_str, X_test, y_test_str, class_order = load_processed_arrays()
y_train = encode_labels(y_train_str, class_order)

# Anti-leakage assertion. X_test and y_test exist only to confirm the
# frozen shapes. They are deleted immediately after the check.
assert X_test.shape[1] == X_train.shape[1], "Feature count mismatch."
assert X_train.shape[0] == 109_188, "Unexpected training row count."
del X_test, y_test_str, y_test

rf_result = train_rf(
    X_train=X_train,
    y_train=y_train,
    class_order=class_order,
)

print("Random Forest training complete.")
print(f"  CV mean Macro F1 : {rf_result['cv_mean_macro_f1']:.4f}")
print(f"  CV std  Macro F1 : {rf_result['cv_std_macro_f1']:.4f}")
print(f"  Fold scores      : {[round(s, 4) for s in rf_result['fold_scores']]}")
print(f"  Best trial       : {rf_result['study'].best_trial.number}")
print(f"  Best params      : {rf_result['best_params']}")
print(f"  Artifacts        : {rf_result['artifacts']}")


Random Forest training complete.
  CV mean Macro F1 : 0.9505
  CV std  Macro F1 : 0.0212
  Fold scores      : [0.9522, 0.9285, 0.9689, 0.925, 0.9781]
  Best trial       : 17
  Best params      : {'n_estimators': 350, 'max_depth': 15, 'min_samples_split': 8, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'criterion': 'entropy', 'class_weight': 'balanced'}
  Artifacts        : {'model': WindowsPath('C:/Users/Rye/OneDrive/Desktop/network-guardian-ids/models/rf_baseline.joblib'), 'meta': WindowsPath('C:/Users/Rye/OneDrive/Desktop/network-guardian-ids/models/rf_baseline.meta.json'), 'trial_csv': WindowsPath('C:/Users/Rye/OneDrive/Desktop/network-guardian-ids/outputs/02_training/optuna_rf_trials.csv'), 'fold_csv': WindowsPath('C:/Users/Rye/OneDrive/Desktop/network-guardian-ids/outputs/02_training/rf_cv_scores.csv'), 'study_db': WindowsPath('C:/Users/Rye/OneDrive/Desktop/network-guardian-ids/outputs/02_training/optuna_study_rf.db')}


In [ ]:
# Cell 6: Random Forest acceptance verification (Amber, Role 3)
# Contract: docs/contracts/role-3-rf-architecture-spec.md section 12.
#
# Reloads the meta sidecar from disk, checks every acceptance threshold,
# and confirms every artifact from section 10 exists. Raises
# AssertionError on any miss so the notebook fails loudly.

import json
import joblib
from src.utils import PATHS

meta_path = PATHS.models / "rf_baseline.meta.json"
model_path = PATHS.models / "rf_baseline.joblib"

assert meta_path.exists(), f"Missing meta sidecar: {meta_path}"
assert model_path.exists(), f"Missing model artifact: {model_path}"

meta = json.loads(meta_path.read_text(encoding="utf-8"))
model = joblib.load(model_path)

checks = [
    ("CV mean Macro F1 >= 0.84",
     meta["cv_mean_macro_f1"] >= 0.84,
     meta["cv_mean_macro_f1"]),
    ("CV std Macro F1 <= 0.025",
     meta["cv_std_macro_f1"] <= 0.025,
     meta["cv_std_macro_f1"]),
    ("OOB Macro F1 within 0.03 of CV mean",
     abs(meta["oob_macro_f1"] - meta["cv_mean_macro_f1"]) <= 0.03,
     meta["oob_macro_f1"]),
    ("u2r recall >= 0.40",
     meta["per_class_recall_cv"]["u2r"] >= 0.40,
     meta["per_class_recall_cv"]["u2r"]),
    ("r2l recall >= 0.65",
     meta["per_class_recall_cv"]["r2l"] >= 0.65,
     meta["per_class_recall_cv"]["r2l"]),
    ("feature_names length == 43",
     len(meta["feature_names"]) == 43,
     len(meta["feature_names"])),
    ("class_order matches canonical",
     list(meta["class_order"]) == ["normal", "dos", "probe", "r2l", "u2r"],
     meta["class_order"]),
    ("fold_scores length == 5",
     len(meta["fold_scores"]) == 5,
     len(meta["fold_scores"])),
    ("per_class_recall_cv has 5 classes",
     set(meta["per_class_recall_cv"].keys()) ==
     {"normal", "dos", "probe", "r2l", "u2r"},
     sorted(meta["per_class_recall_cv"].keys())),
    ("model exposes 5 classes",
     list(model.classes_) == [0, 1, 2, 3, 4],
     list(model.classes_)),
    ("oob_score enabled",
     bool(getattr(model, "oob_score", False)),
     getattr(model, "oob_score", False)),
]

print("Random Forest acceptance checks")
print("-" * 60)
failed = []
for label, passed, value in checks:
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {label}  (value={value})")
    if not passed:
        failed.append(label)

required_artifacts = [
    PATHS.models / "rf_baseline.joblib",
    PATHS.models / "rf_baseline.meta.json",
    PATHS.outputs / "02_training" / "optuna_study_rf.db",
    PATHS.outputs / "02_training" / "optuna_rf_trials.csv",
    PATHS.outputs / "02_training" / "rf_cv_scores.csv",
]
print("\nArtifact presence")
print("-" * 60)
for p in required_artifacts:
    exists = p.exists()
    print(f"  [{'PASS' if exists else 'FAIL'}] {p}")
    if not exists:
        failed.append(str(p))

if failed:
    raise AssertionError(
        "RF acceptance criteria not met. Report to Kit before proceeding. "
        f"Failures: {failed}"
    )
print("\nAll RF acceptance criteria met. Ready for handoff note.")


Random Forest acceptance checks
------------------------------------------------------------
  [PASS] CV mean Macro F1 >= 0.84  (value=0.9505402684192411)
  [FAIL] CV std Macro F1 <= 0.02  (value=0.021177682093871768)
  [PASS] OOB Macro F1 within 0.03 of CV mean  (value=0.9495747664045213)
  [PASS] u2r recall >= 0.40  (value=0.8205128205128205)
  [PASS] r2l recall >= 0.65  (value=0.9586114819759679)
  [PASS] feature_names length == 43  (value=43)
  [PASS] class_order matches canonical  (value=['normal', 'dos', 'probe', 'r2l', 'u2r'])
  [PASS] fold_scores length == 5  (value=5)
  [PASS] per_class_recall_cv has 5 classes  (value=['dos', 'normal', 'probe', 'r2l', 'u2r'])
  [PASS] model exposes 5 classes  (value=[0, 1, 2, 3, 4])
  [PASS] oob_score enabled  (value=True)

Artifact presence
------------------------------------------------------------
  [PASS] C:\Users\Rye\OneDrive\Desktop\network-guardian-ids\models\rf_baseline.joblib
  [PASS] C:\Users\Rye\OneDrive\Desktop\network-guardian-id

AssertionError: RF acceptance criteria not met. Report to Kit before proceeding. Failures: ['CV std Macro F1 <= 0.02']

In [ ]:
# XGBoost training (Kenneth, Role 4)
# Contract: docs/contracts/role-4-training-execution-spec.md
# Estimated wall time: 4 to 6 hours

xgb_result = train_xgb(
    X_train=X_train,
    y_train=y_train,
    class_order=class_order,
    n_trials=30,
    n_folds=5,
)
print("XGBoost CV Macro F1:", xgb_result["cv_mean_macro_f1"])


In [ ]:
# XGBoost artifact verification (Kenneth, Role 4)

assert (PATHS.models / "xgb_optuna.joblib").is_file(), "XGBoost model missing"
assert (PATHS.models / "xgb_optuna.meta.json").is_file(), "XGBoost meta missing"
print("XGBoost artifacts verified.")


In [4]:
# TabNet training (Dwayne, Role 5)
# Contract: docs/contracts/role-5-training-execution-spec.md
# Estimated wall time: 6 to 8 hours

tabnet_result = train_tabnet(
    X_train=X_train,
    y_train=y_train,
    class_order=class_order,
    n_trials=20,
    n_folds=3,
)
print("TabNet CV Macro F1:", tabnet_result["cv_mean_macro_f1"])



Early stopping occurred at epoch 15 with best_epoch = 5 and best_val_logloss = 0.02548

Early stopping occurred at epoch 23 with best_epoch = 13 and best_val_logloss = 0.01474
Stop training because you reached max_epochs = 30 with best_epoch = 28 and best_val_logloss = 0.013

Early stopping occurred at epoch 17 with best_epoch = 7 and best_val_logloss = 0.02068

Early stopping occurred at epoch 20 with best_epoch = 10 and best_val_logloss = 0.01469

Early stopping occurred at epoch 23 with best_epoch = 13 and best_val_logloss = 0.01265
Stop training because you reached max_epochs = 30 with best_epoch = 28 and best_val_logloss = 0.01259
Stop training because you reached max_epochs = 30 with best_epoch = 23 and best_val_logloss = 0.00804

Early stopping occurred at epoch 26 with best_epoch = 16 and best_val_logloss = 0.01032

Early stopping occurred at epoch 15 with best_epoch = 5 and best_val_logloss = 0.01447

Early stopping occurred at epoch 17 with best_epoch = 7 and best_val_loglos

In [4]:
# TabNet artifact verification (Dwayne, Role 5)

from pathlib import Path
import sys

ROOT = Path(r"C:\SCHOOL\I.S. PROJECT\network-guardian-ids")
sys.path.insert(0, str(ROOT))

from src.utils import PATHS

assert (PATHS.models / "tabnet_best.zip").is_file(), "TabNet model missing"
assert (PATHS.models / "tabnet_best.meta.json").is_file(), "TabNet meta missing"

print("TabNet artifacts verified.")

TabNet artifacts verified.


In [ ]:
# Cross-model comparison (Kit, Role 1)
# Run after all three training cells complete.

rows = []

for name, meta_file in [
    ("Random Forest", "rf_baseline.meta.json"),
    ("XGBoost",       "xgb_optuna.meta.json"),
    ("TabNet",        "tabnet_best.meta.json"),
]:
    meta_path = PATHS.models / meta_file
    if not meta_path.is_file():
        rows.append({"model": name, "status": "MISSING"})
        continue
    m = json.load(open(meta_path))
    pc = m.get("per_class_recall_train", {})
    rows.append({
        "model":            name,
        "cv_mean_macro_f1": m["cv_mean_macro_f1"],
        "cv_std_macro_f1":  m["cv_std_macro_f1"],
        "u2r_recall":       pc.get("u2r"),
        "r2l_recall":       pc.get("r2l"),
        "probe_recall":     pc.get("probe"),
        "dos_recall":       pc.get("dos"),
        "normal_recall":    pc.get("normal"),
        "status":           "OK",
    })

df = pd.DataFrame(rows)
display(df)

comparison_path = PATHS.outputs / "02_training" / "model_comparison.csv"
comparison_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(comparison_path, index=False)
print("Saved:", comparison_path)


In [ ]:
# Final summary (Kit, Role 1)

print("=" * 68)
print("Notebook 02 Summary")
print("=" * 68)
for name, meta_file in [
    ("Random Forest", "rf_baseline.meta.json"),
    ("XGBoost",       "xgb_optuna.meta.json"),
    ("TabNet",        "tabnet_best.meta.json"),
]:
    meta_path = PATHS.models / meta_file
    if meta_path.is_file():
        m = json.load(open(meta_path))
        print(f"{name:<14} CV Macro F1 = {m['cv_mean_macro_f1']:.4f}")
    else:
        print(f"{name:<14} MISSING")
print("=" * 68)
